# Fusión Simple: Aditivos + Alimentos

## 📋 Objetivo

Fusionar:
1. **Dataset de aditivos**: `id` + `traffic_light` (🟢 SEGURO, 🟡 PRECAUCIÓN, 🔴 EVITABLE)
2. **Dataset de alimentos**: `nutriscore_grade`, `nova_group` + one-hot encoding (columnas = aditivos)

### 🎯 Resultado
Conteos de aditivos por categoría de seguridad:
- Conteo aditivos seguros (🟢)
- Conteo aditivos precaución (🟡)
- Conteo aditivos evitables (🔴)
- Total de aditivos

## 1️⃣ Cargar Datasets

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')

print("📖 Cargando datasets...\n")

# Cargar datasets
dataset_aditivos = pd.read_csv(DATA_DIR / 'ssi_final_con_semaforo.csv')
dataset_alimentos = pd.read_csv(DATA_DIR / 'dataset_800k_aditivos.csv')

print(f"✅ Aditivos: {len(dataset_aditivos)}")
print(f"   Columnas: {dataset_aditivos.columns.tolist()[:5]}")
print(f"   Traffic light values: {dataset_aditivos['traffic_light'].unique()}")

print(f"\n✅ Alimentos: {len(dataset_alimentos):,}")
print(f"   Columnas clave: nutriscore_grade, nova_group")

# Identificar columnas de aditivos (corresponden a los 'id' de aditivos)
columnas_aditivos = [col for col in dataset_alimentos.columns 
                      if col not in ['product_name', 'nutriscore_grade', 'nova_group']]
print(f"   Columnas de aditivos (one-hot): {len(columnas_aditivos)}")
print(f"   Primeras: {columnas_aditivos[:10]}")

📖 Cargando datasets...

✅ Aditivos: 651
   Columnas: ['id', 'e_code', 'nombre', 'ssi', 'confidence']
   Traffic light values: ['SEGURO' 'EVITAR' 'PRECAUCION']

✅ Alimentos: 837,548
   Columnas clave: nutriscore_grade, nova_group
   Columnas de aditivos (one-hot): 589
   Primeras: ['ca:carbonat-acid-de-sodi-i-difosfat-de-disodi', 'de:gärungskohlensäure', 'de:natürliche-quellkohlensäure', 'de:quellkohlensäure', 'en:amino-acids', 'en:ammonium-and-sodium-carbonates', 'en:di-and-triphosphates', 'en:diphosphates-and-sodium-carbonates', 'en:e100', 'en:e1000']


## 2️⃣ Crear Mapeo: ID Aditivo → Traffic Light

In [2]:
print("\n" + "="*80)
print("CREAR MAPEO ADITIVOS → TRAFFIC LIGHT")
print("="*80)

# Ver distribución del traffic light
print(f"\n📊 Distribución traffic light:")
print(dataset_aditivos['traffic_light'].value_counts())

# Crear mapeo: id (nombre columna) → traffic_light
mapeo_aditivo_trafficlight = dict(zip(
    dataset_aditivos['id'],
    dataset_aditivos['traffic_light']
))

print(f"\n✅ Mapeo creado: {len(mapeo_aditivo_trafficlight)} aditivos")
print(f"   Ejemplos:")
for id_aditivo, trafficlight in list(mapeo_aditivo_trafficlight.items())[:5]:
    print(f"      {id_aditivo} → {trafficlight}")


CREAR MAPEO ADITIVOS → TRAFFIC LIGHT

📊 Distribución traffic light:
traffic_light
SEGURO        475
EVITAR         92
PRECAUCION     84
Name: count, dtype: int64

✅ Mapeo creado: 651 aditivos
   Ejemplos:
      en:e333ii → SEGURO
      en:e321 → EVITAR
      en:e345 → SEGURO
      en:e303 → SEGURO
      en:e1104 → EVITAR


## 3️⃣ Contar Aditivos por Categoría

In [14]:
"""
CELDA OPTIMIZADA CON JOBLIB (FUNCIONA EN JUPYTER)
Usa todos hilos del PC para colapsar el one hot encoding
"""

print("\n" + "="*80)
print("CONTAR ADITIVOS POR CATEGORÍA (OPTIMIZADO - MULTICORE CON JOBLIB)")
print("="*80)

from joblib import Parallel, delayed
import time

print(f"\n⚙️  Procesando {len(dataset_alimentos):,} alimentos...")
print(f"   Analizando {len(columnas_aditivos)} columnas de aditivos")

# ════════════════════════════════════════════════════════════════════════════
# FUNCIÓN PARA PROCESAR UN ALIMENTO
# ════════════════════════════════════════════════════════════════════════════

def procesar_alimento(idx, row):
    """
    Procesa UN alimento y retorna conteos de aditivos
    
    Args:
        idx: índice del alimento
        row: la fila del alimento
    
    Returns:
        (idx, seguros, precaucion, evitables)
    """
    seguros = 0
    precaucion = 0
    evitables = 0
    
    # Para cada aditivo (columna = id del aditivo)
    for aditivo_id in columnas_aditivos:
        # Si el alimento contiene este aditivo (valor 1)
        if row[aditivo_id] == 1:
            # Buscar el traffic light de este aditivo
            trafficlight = mapeo_aditivo_trafficlight.get(aditivo_id)
            
            if trafficlight == 'SEGURO':
                seguros += 1
            elif trafficlight == 'PRECAUCION':
                precaucion += 1
            elif trafficlight == 'EVITAR':
                evitables += 1
    
    return (idx, seguros, precaucion, evitables)

# ════════════════════════════════════════════════════════════════════════════
# PROCESAMIENTO PARALELO CON JOBLIB
# ════════════════════════════════════════════════════════════════════════════

print(f"\n⚡ Detectando CPUs disponibles...\\n")

inicio = time.time()

# Usar joblib.Parallel - FUNCIONA EN JUPYTER
# n_jobs=-1 usa TODOS los cores disponibles
resultados = Parallel(n_jobs=-1, verbose=10)(
    delayed(procesar_alimento)(idx, row) 
    for idx, row in dataset_alimentos.iterrows()
)

tiempo_procesamiento = time.time() - inicio
print(f"\n✅ Procesamiento paralelo completado en {tiempo_procesamiento:.1f}s")
print(f"   Velocidad: {len(dataset_alimentos) / tiempo_procesamiento / 1000:.1f}k alimentos/segundo")

# ════════════════════════════════════════════════════════════════════════════
# ASIGNAR RESULTADOS AL DATAFRAME
# ════════════════════════════════════════════════════════════════════════════

print(f"\n   Asignando resultados al dataset...")

# Inicializar contadores
dataset_alimentos['conteo_aditivos_seguros'] = 0
dataset_alimentos['conteo_aditivos_precaucion'] = 0
dataset_alimentos['conteo_aditivos_evitables'] = 0

# Asignar resultados
for idx, seguros, precaucion, evitables in resultados:
    dataset_alimentos.at[idx, 'conteo_aditivos_seguros'] = seguros
    dataset_alimentos.at[idx, 'conteo_aditivos_precaucion'] = precaucion
    dataset_alimentos.at[idx, 'conteo_aditivos_evitables'] = evitables

print(f"✅ Conteos completados y asignados")

# ════════════════════════════════════════════════════════════════════════════
# VALIDACIÓN
# ════════════════════════════════════════════════════════════════════════════

print(f"\n✅ VALIDACIÓN:")
print(f"   Registros procesados: {len(resultados):,}")

total_aditivos = dataset_alimentos['conteo_aditivos_seguros'] + dataset_alimentos['conteo_aditivos_precaucion'] + dataset_alimentos['conteo_aditivos_evitables']

print(f"   Media aditivos por alimento: {total_aditivos.mean():.2f}")
print(f"   Alimentos sin aditivos: {(total_aditivos == 0).sum():,}")
print(f"   Alimentos con aditivos: {(total_aditivos > 0).sum():,}")


CONTAR ADITIVOS POR CATEGORÍA (OPTIMIZADO - MULTICORE CON JOBLIB)

⚙️  Procesando 837,548 alimentos...
   Analizando 589 columnas de aditivos

⚡ Detectando CPUs disponibles...\n


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    9.3s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19092901279432148s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    9.4s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    9.4s
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:    9.4s
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    9.4s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:    9.4s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.052915334701538086s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:    9.5s
[Parallel(n_jobs=-1)]: Done 242 tasks      | ela


✅ Procesamiento paralelo completado en 398.8s
   Velocidad: 2.1k alimentos/segundo

   Asignando resultados al dataset...
✅ Conteos completados y asignados

✅ VALIDACIÓN:
   Registros procesados: 837,548
   Media aditivos por alimento: 2.30
   Alimentos sin aditivos: 342,716
   Alimentos con aditivos: 494,832


## 4️⃣ Calcular Total y Limpiar

In [15]:
print("\n" + "="*80)
print("LIMPIAR Y FINALIZAR")
print("="*80)

# Calcular total de aditivos
dataset_alimentos['total_aditivos'] = (
    dataset_alimentos['conteo_aditivos_seguros'] +
    dataset_alimentos['conteo_aditivos_precaucion'] +
    dataset_alimentos['conteo_aditivos_evitables']
)

print(f"\n✅ Total calculado")

# Seleccionar columnas finales (sin one-hot encoding)
columnas_finales = [
    'product_name',
    'nutriscore_grade',
    'nova_group',
    'conteo_aditivos_seguros',
    'conteo_aditivos_precaucion',
    'conteo_aditivos_evitables',
    'total_aditivos'
]

df_final = dataset_alimentos[columnas_finales].copy()

print(f"\n✅ Dataset final: {len(df_final):,} alimentos")
print(f"   Columnas: {len(df_final.columns)}")
print(f"   (Eliminado: one-hot encoding de aditivos)")


LIMPIAR Y FINALIZAR

✅ Total calculado

✅ Dataset final: 837,548 alimentos
   Columnas: 7
   (Eliminado: one-hot encoding de aditivos)


## 5️⃣ Estadísticas y Validación

In [16]:
print("\n" + "="*80)
print("ESTADÍSTICAS FINALES")
print("="*80)

print(f"\n📊 Nutriscore:")
print(f"   Media: {df_final['nutriscore_grade'].mean():.2f}")
print(f"   Rango: {df_final['nutriscore_grade'].min():.0f} - {df_final['nutriscore_grade'].max():.0f}")
print(f"   Distribución:")
for valor in sorted(df_final['nutriscore_grade'].unique()):
    count = (df_final['nutriscore_grade'] == valor).sum()
    pct = (count / len(df_final)) * 100
    print(f"      {valor:.0f}: {count:7,} ({pct:5.1f}%)")

print(f"\n📊 NOVA:")
print(f"   Media: {df_final['nova_group'].mean():.2f}")
print(f"   Valores únicos: {sorted(df_final['nova_group'].unique())}")

print(f"\n📊 Aditivos:")
print(f"   🟢 Seguros      - Media: {df_final['conteo_aditivos_seguros'].mean():.2f}")
print(f"   🟡 Precaución   - Media: {df_final['conteo_aditivos_precaucion'].mean():.2f}")
print(f"   🔴 Evitables    - Media: {df_final['conteo_aditivos_evitables'].mean():.2f}")
print(f"   📊 Total        - Media: {df_final['total_aditivos'].mean():.2f}")

print(f"\n📊 CONTEO TOTAL DE EVITABLES:")
total_evitables = df_final['conteo_aditivos_evitables'].sum()
print(f"   🔴 Total de aditivos evitables en toda la BD: {total_evitables:,}")
pct_evitables = (total_evitables / df_final['total_aditivos'].sum()) * 100
print(f"   Porcentaje del total: {pct_evitables:.1f}%")


ESTADÍSTICAS FINALES

📊 Nutriscore:
   Media: 3.40
   Rango: 1 - 5
   Distribución:
      1: 126,005 ( 15.0%)
      2:  92,342 ( 11.0%)
      3: 177,776 ( 21.2%)
      4: 202,284 ( 24.2%)
      5: 239,141 ( 28.6%)

📊 NOVA:
   Media: 3.31
   Valores únicos: [1.0, 2.0, 3.0, 4.0]

📊 Aditivos:
   🟢 Seguros      - Media: 0.99
   🟡 Precaución   - Media: 0.42
   🔴 Evitables    - Media: 0.89
   📊 Total        - Media: 2.30

📊 CONTEO TOTAL DE EVITABLES:
   🔴 Total de aditivos evitables en toda la BD: 747,073
   Porcentaje del total: 38.8%


## 6️⃣ Guardar Resultado Final

In [ ]:
print("\n" + "="*80)
print("GUARDAR DATASET FINAL")
print("="*80)

# Guardar
output_file = DATA_DIR / 'alimentos_con_semaforo_final.csv'
df_final.to_csv(output_file, index=False)

print(f"\n💾 Archivo guardado: {output_file}")
print(f"   Tamaño: {output_file.stat().st_size / (1024**2):.2f} MB")
print(f"   Registros: {len(df_final):,}")
print(f"   Columnas: {len(df_final.columns)}")

print(f"\n🎯 COLUMNAS FINALES:")
print(f"   ✅ product_name")
print(f"   ✅ nutriscore_grade (1-5)")
print(f"   ✅ nova_group (1-4)")
print(f"   ✅ conteo_aditivos_seguros (🟢)")
print(f"   ✅ conteo_aditivos_precaucion (🟡)")
print(f"   ✅ conteo_aditivos_evitables (🔴)")
print(f"   ✅ total_aditivos")


GUARDAR DATASET FINAL

💾 Archivo guardado: ../data/alimentos_con_semaforo_final2.csv
   Tamaño: 24.44 MB
   Registros: 837,548
   Columnas: 7

🎯 COLUMNAS FINALES:
   ✅ product_name
   ✅ nutriscore_grade (1-5)
   ✅ nova_group (1-4)
   ✅ conteo_aditivos_seguros (🟢)
   ✅ conteo_aditivos_precaucion (🟡)
   ✅ conteo_aditivos_evitables (🔴)
   ✅ total_aditivos
